In [2]:
# ==========================================
# Importar bibliotecas
# ==========================================
import os
import json
import pickle
import numpy as np
import faiss
from openai import OpenAI
from dotenv import load_dotenv

# ==========================================
# Configuração do ambiente e cliente OpenAI
# ==========================================
load_dotenv()  # Carregar variáveis do arquivo .env
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # Inicializar cliente OpenAI

# ==========================================
# Carregar os chunks e metadados
# ==========================================
with open("metadata.pkl", "rb") as f:
    embeddings_list = pickle.load(f)

with open(os.getenv("PROCESSED_JSON"), "r", encoding="utf-8") as f:
    chunks = json.load(f)

index = faiss.read_index("embeddings.index")

# ==========================================
# Gerar embeddings da query e buscar no FAISS
# ==========================================
import faiss
import pickle

# Carregar índice FAISS e metadados
index = faiss.read_index("embeddings.index")
with open("metadata.pkl", "rb") as f:
    embeddings_list = pickle.load(f)

def gerar_embedding(texto):
    resp = client.embeddings.create(
        model="text-embedding-3-large",
        input=texto
    )
    return np.array(resp.data[0].embedding, dtype="float32")

def buscar_chunks(query, top_k=5):
    emb_query = gerar_embedding(query)
    D, I = index.search(emb_query.reshape(1, -1), top_k)  # D = distâncias, I = índices
    resultados = []
    for idx, dist in zip(I[0], D[0]):
        resultados.append({
            "chunk_number": embeddings_list[idx]["chunk_number"],
            "source_file": embeddings_list[idx]["source_file"],
            "page_number": embeddings_list[idx]["page_number"],
            "distancia": float(dist),
            "texto": chunks[idx]["text"]
        })
    return resultados

# ==========================================
# Testar consultas
# ==========================================
avaliacoes = [
    {"query": "Qual contrato tem a empresa com sede na Rua Alfred Charvet", "relevante": [0]},
    {"query": "ANDREY BARRETO MOLINARI COMERCIO", "relevante": [27, 45]},
    {"query": "Qual empresa é representada pelo sócio administrador Jhonny Cézar de Jesus Falavinha", "relevante": [0, 25]}, 
    {"query": "Jhonny Cézar de Jesus Falavinha", "relevante": [0, 25]},
    {"query": "Contrato que tenha o CNPJ 76.443.340/0001-59", "relevante": [47, 48]}
]   

for caso in avaliacoes:
    resultados = buscar_chunks(caso["query"], top_k=5)
    print(f"Query: {caso['query']}")
    for r in resultados:
        print(f"Chunk {r['chunk_number']} - Distância: {r['distancia']:.4f} - Arquivo: {r['source_file']}")
    print("\n")


Query: Qual contrato tem a empresa com sede na Rua Alfred Charvet
Chunk 0 - Distância: 0.9854 - Arquivo: AJJ.SERVICOS.CONTABEIS.LTDA.-.Contrato.de.Empresa.Associada.1.pdf
Chunk 25 - Distância: 1.0027 - Arquivo: ANDREY.BARRETO.MOLINARI.COMERCIO.SERVICOS.E.FABRICACAO.DE.MAQUINAS.-.Contrato.de.Empresa.Associada.pdf
Chunk 44 - Distância: 1.0158 - Arquivo: ARCIMOL.-.PRE.MOLDADOS.E.CONSTRUTORA.DE.OBRAS.LTDA.-.Contrato.de.Empresa.Associada.pdf
Chunk 27 - Distância: 1.0287 - Arquivo: ANDREY.BARRETO.MOLINARI.COMERCIO.SERVICOS.E.FABRICACAO.DE.MAQUINAS.-.Contrato.de.Empresa.Associada.pdf
Chunk 58 - Distância: 1.0409 - Arquivo: ARCIMOL.-.PRE.MOLDADOS.E.CONSTRUTORA.DE.OBRAS.LTDA.-.Contrato.de.Empresa.Associada.pdf


Query: ANDREY BARRETO MOLINARI COMERCIO
Chunk 41 - Distância: 1.0767 - Arquivo: ANDREY.BARRETO.MOLINARI.COMERCIO.SERVICOS.E.FABRICACAO.DE.MAQUINAS.-.Contrato.de.Empresa.Associada.pdf
Chunk 25 - Distância: 1.1098 - Arquivo: ANDREY.BARRETO.MOLINARI.COMERCIO.SERVICOS.E.FABRICACAO.DE.MAQUIN